# nbjit.jl Demo: Edit-Aware Partial Compilation for Notebooks

This notebook demonstrates nbjit.jl's core features:

1. **Manual `@hole` mode** — explicit annotation of frequently-changing expressions
2. **Automatic mode (LCS)** — statement-level hash diff detects what changed
3. **Automatic mode (GumTree)** — AST-level tree diff for finer-grained change detection
4. **Inter-cell dependency tracking** — the system knows which cells depend on which

The key idea: when you edit and re-execute code, nbjit.jl reuses previous compilation results where safe, instead of recompiling from scratch.

In [1]:
include("../src/ijulia_integration.jl")
using .IJuliaIntegration

┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_dict_new :: Tuple{}` in module `Main.IJuliaIntegration`
└ @ Base.Docs docs/Docs.jl:249
┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_dict_new_with_pairs :: Tuple{Ptr{Nothing}, Ptr{Nothing}, Int64}` in module `Main.IJuliaIntegration`
└ @ Base.Docs docs/Docs.jl:249
┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_dict_getindex :: Tuple{Ptr{Nothing}, Ptr{Nothing}}` in module `Main.IJuliaIntegration`
└ @ Base.Docs docs/Docs.jl:249
┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_dict_setindex! :: Tuple{Ptr{Nothing}, Ptr{Nothing}, Ptr{Nothing}}` in module `Main.IJuliaIntegration`
└ @ Base.Docs docs/Docs.jl:249
┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_symbol_from_cstr :: Tuple{Ptr{UInt8}}` in module `Main.IJuliaIntegration`
└ @ Base.Docs docs/Docs.jl:249
┌ Warning: Replacing docs for `Main.IJuliaIntegration.nbjit_box_int64 :: Tuple{Int64}` in module `Main.IJuliaIntegration`
└ @ 

## Part 1: Manual `@hole` Mode

Use `@jit` with `@hole` to mark expressions that change frequently.
The main block (everything outside `@hole`) is compiled once and reused.
Only holes are recompiled when their content changes.

In [ ]:
# First execution: everything is compiled
@jit begin
    base = 1000
    @hole adjustment = 42
    result = base + adjustment
end

In [ ]:
# Change ONLY the @hole value.
# Main is reused (cached), only the hole is recompiled.
@jit begin
    base = 1000
    @hole adjustment = 99
    result = base + adjustment
end

In [ ]:
# Change the main block structure.
# This triggers full recompilation (main + holes).
@jit begin
    base = 1000
    multiplier = 2
    @hole adjustment = 99
    result = base * multiplier + adjustment
end

### Multiple Holes

Each hole is compiled to a separate shared library.
When only one hole changes, only that hole is recompiled.

In [ ]:
# Two holes: both compiled on first run
session = IJuliaIntegration.NotebookSession()

code_v1 = quote
    base = 100
    @hole lr = 1
    temp = base + lr
    @hole epochs = 10
    result = temp * epochs
end

res1 = IJuliaIntegration.run_cell!(session, code_v1; cell_id="multi_hole")
println("v1: result=$(res1.result), rebuilt_main=$(res1.rebuilt_main), recompiled=$(res1.recompiled_holes)")

In [ ]:
# Change only the second hole (epochs)
code_v2 = quote
    base = 100
    @hole lr = 1
    temp = base + lr
    @hole epochs = 50
    result = temp * epochs
end

res2 = IJuliaIntegration.run_cell!(session, code_v2; cell_id="multi_hole")
println("v2: result=$(res2.result), rebuilt_main=$(res2.rebuilt_main), recompiled=$(res2.recompiled_holes)")
println("   -> Only hole 2 recompiled! Main and hole 1 reused.")

## Part 2: Automatic Mode with GumTree Tree Diff

With automatic hole detection, you don't need `@hole` annotations.
Just use `@jit` and the system detects what changed automatically.

nbjit.jl provides two diff algorithms:
- **LCS** (default): statement-level hash comparison, fast and simple
- **GumTree**: AST-level tree diff, detects changes at the node level

GumTree uses the algorithm from Falleri et al. (ASE 2014):
1. **Top-down**: match isomorphic subtrees greedily from largest to smallest
2. **Bottom-up**: recover remaining matches using structural similarity
3. **Change detection**: a statement is changed if any leaf node differs in value

Here we use `diff_algorithm=:gumtree`:

In [2]:
# Create a session with GumTree diff algorithm
session_gt = IJuliaIntegration.NotebookSession(diff_algorithm=:gumtree)

# First execution: full compilation (no previous version to diff against)
code_gt_v1 = quote
    n = 1000000
    total = 0
    for i in 1:n
        total = total + i
    end
    total
end

res_gt1 = IJuliaIntegration.run_cell!(session_gt, code_gt_v1; cell_id="gt_demo")
println("v1: result=$(res_gt1.result)")
println("    $(res_gt1.dylib_info)")

v1: result=500000500000
    Main: nbjit_main_func_1_PVA6YluM.so, auto (first run)


In [3]:
# Re-execute with same code: fully cached, no recompilation
res_gt1b = IJuliaIntegration.run_cell!(session_gt, code_gt_v1; cell_id="gt_demo")
println("v1 (re-run): result=$(res_gt1b.result)")
println("    $(res_gt1b.dylib_info)")
println("    rebuilt_main=$(res_gt1b.rebuilt_main), recompiled_holes=$(res_gt1b.recompiled_holes)")

v1 (re-run): result=500000500000
    Main: nbjit_main_func_1_PVA6YluM.so, auto (cached)
    rebuilt_main=false, recompiled_holes=Int64[]


In [4]:
# Change just the loop bound (n = 1000000 -> n = 2000000)
# GumTree matches the tree structure (for loop, total = total + i, etc.)
# and detects that only the leaf value 1000000 -> 2000000 changed.
code_gt_v2 = quote
    n = 2000000
    total = 0
    for i in 1:n
        total = total + i
    end
    total
end

res_gt2 = IJuliaIntegration.run_cell!(session_gt, code_gt_v2; cell_id="gt_demo")
println("v2: result=$(res_gt2.result)")
println("    $(res_gt2.dylib_info)")
println("    -> GumTree detected the changed statement as a hole!")

Main structure changed - full recompilation required
v2: result=2000001000000
    Main: nbjit_main_func_3_hSdBsW6X.so, auto (1 hole(s) detected)
    -> GumTree detected the changed statement as a hole!


In [5]:
# Change the same statement again (n = 2000000 -> n = 5000000)
# Main is reused, only the hole is recompiled.
code_gt_v3 = quote
    n = 5000000
    total = 0
    for i in 1:n
        total = total + i
    end
    total
end

res_gt3 = IJuliaIntegration.run_cell!(session_gt, code_gt_v3; cell_id="gt_demo")
println("v3: result=$(res_gt3.result)")
println("    $(res_gt3.dylib_info)")
println("    recompiled_holes=$(res_gt3.recompiled_holes)")

Holes changed: [1] - recompiling holes only (reusing main dylib)
v3: result=12500002500000
    Main: nbjit_main_func_3_hSdBsW6X.so, auto (1 hole(s) detected)
    recompiled_holes=[1]


### Comparing LCS vs GumTree

Both algorithms detect the same changed statements in most cases.
GumTree provides additional precision: it knows exactly which AST nodes changed,
not just that the statement hash differs.

In [6]:
old_code = quote
    x = 10
    y = x * 2 + 100
    z = y + 1
end
new_code = quote
    x = 10
    y = x * 2 + 200
    z = y + 1
end

# LCS: compares statement hashes
lcs_main, lcs_holes, _ = IJuliaIntegration.auto_prepare_split(old_code, new_code)
println("LCS:     $(length(lcs_holes)) hole(s) detected")

# GumTree: compares AST structure at node level
gt_main, gt_holes, _ = IJuliaIntegration.gumtree_prepare_split(old_code, new_code)
println("GumTree: $(length(gt_holes)) hole(s) detected")

# Both detect 1 hole (y = x * 2 + 200)
# But GumTree additionally knows that only the literal 100->200 changed,
# while the rest of the statement structure (x * 2 + ...) is identical.

# We can see this via changed_statement_indices:
changed = IJuliaIntegration.changed_statement_indices(old_code, new_code)
println("GumTree changed statement indices: $changed")

LCS:     1 hole(s) detected
GumTree: 1 hole(s) detected
GumTree changed statement indices: Set([2])


## Part 3: Inter-Cell Dependency Tracking

nbjit.jl tracks which symbols each cell defines and references.
When a cell is re-executed, the system reports which downstream cells
have become stale and need re-execution.

In [7]:
session_deps = IJuliaIntegration.NotebookSession()

# Cell A: defines x
code_a = quote
    x = 10
    @hole delta = 1
    x = x + delta
end

# Cell B: uses x, defines y
code_b = quote
    y = x * 2
    @hole scale = 3
    y = y + scale
end

# Cell C: uses y
code_c = quote
    z = y + 100
    @hole offset = 5
    z = z + offset
end

# Register all cells
res_a = IJuliaIntegration.run_cell!(session_deps, code_a; cell_id="A")
# B and C can't compile (they reference cross-cell variables),
# but we can register them in the dependency graph manually:
IJuliaIntegration.update_cell!(session_deps.dep_graph, "B", code_b)
IJuliaIntegration.update_cell!(session_deps.dep_graph, "C", code_c)

println("Dependency graph:")
println("  A defines: $(IJuliaIntegration.get_cell_definitions(session_deps.dep_graph, "A"))")
println("  B depends on: $(IJuliaIntegration.get_upstream(session_deps.dep_graph, "B"))")
println("  C depends on: $(IJuliaIntegration.get_upstream(session_deps.dep_graph, "C"))")
println("  A's downstream: $(IJuliaIntegration.get_downstream(session_deps.dep_graph, "A"))")

Dependency graph:
  A defines: Set([:delta, :x])
  B depends on: Set(["A"])
  C depends on: Set(["B"])
  A's downstream: Set(["B"])


In [8]:
# Re-execute Cell A with a change.
# The system reports that B and C are now stale.
code_a_v2 = quote
    x = 20
    @hole delta = 1
    x = x + delta
end

res_a2 = IJuliaIntegration.run_cell!(session_deps, code_a_v2; cell_id="A")
println("After re-executing A:")
println("  Stale downstream cells: $(res_a2.stale_cells)")
println("  -> B and C need re-execution because they depend on x (defined by A)")

Main structure changed - full recompilation required
After re-executing A:
  Stale downstream cells: ["B", "C"]
  -> B and C need re-execution because they depend on x (defined by A)


## Part 4: Performance Comparison

nbjit.jl compiles notebook cells to native code via LLVM.
For numerical loops, this is dramatically faster than Julia's interpreter.

In [9]:
# nbjit.jl: compiled to native code
println("=== nbjit.jl (native compilation) ===")
@time @jit begin
    @hole iterations = 10000000
    result = 0
    for i in 1:iterations
        result = result + i
    end
    result
end

=== nbjit.jl (native compilation) ===


Cell In[9]: main func_9 (recompiled) [Dylib (separate compilation)]
  Main: nbjit_main_func_9_laLT3DHM.so, Holes: 1
  hole 1 -> func_8 (recompiled)
  result: 50000005000000


  0.207099 seconds (378.62 k allocations: 18.548 MiB, 84.89% compilation time: 3% of which was recompilation)


Cell In[9]: main func_9 (recompiled) [Dylib (separate compilation)]
  Main: nbjit_main_func_9_laLT3DHM.so, Holes: 1
  hole 1 -> func_8 (recompiled)
  result: 50000005000000


In [10]:
# Vanilla Julia: interpreted in global scope
println("=== Vanilla Julia (global scope) ===")
@time begin
    iterations = 10000000
    result = 0
    for i in 1:iterations
        result = result + i
    end
    result
end

=== Vanilla Julia (global scope) ===
  2.475786 seconds (40.00 M allocations: 762.924 MiB, 2.88% gc time)


50000005000000

In [ ]:
# Now change only the iteration count.
# nbjit.jl reuses the main loop code and only recompiles the parameter.
println("=== nbjit.jl (hole-only recompilation) ===")
@time @jit begin
    @hole iterations = 50000000
    result = 0
    for i in 1:iterations
        result = result + i
    end
    result
end

## Summary

| Feature | What it does |
|---|---|
| `@jit` + `@hole` | Manual annotation: mark frequently-changing expressions |
| `@jit` (no annotation, LCS) | Automatic mode: statement-level hash diff |
| `@jit` (no annotation, GumTree) | Automatic mode: AST-level tree diff for finer precision |
| `diff_algorithm=:gumtree` | Opt-in per session: `NotebookSession(diff_algorithm=:gumtree)` |
| Dependency tracking | Knows which cells depend on which; reports stale cells |
| Native compilation | LLVM-compiled numerical code runs orders of magnitude faster |